# 05 — Baseline Models: Individual Base Learner Evaluation

This notebook trains each base learner individually with cross-validation, reports metrics, and compares models visually.

**Base Learners:**
- Random Forest
- XGBoost
- LightGBM
- SVM (RBF kernel)
- KNN

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold

from src.config import DATASETS, SEED, STACKING_PARAMS
from src.data.loader import load_raw_dataset, get_target_column
from src.data.preprocessor import MediSensePreprocessor
from src.data.splitter import stratified_split
from src.models.base_learners import get_base_learners
from src.evaluation.metrics import compute_metrics, format_metrics

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## Helper: Prepare Dataset

In [ ]:
def prepare_dataset(dataset_name):
    """Load, clean, split, and preprocess a dataset."""
    df = load_raw_dataset(dataset_name)
    target_col = get_target_column(dataset_name)
    
    if dataset_name == 'heart':
        df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
        df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
        df['target'] = (df['target'] > 0).astype(int)
    elif dataset_name == 'liver':
        df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})
    
    train_df, val_df, test_df = stratified_split(df, target_col)
    
    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col].values
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col].values
    
    preprocessor = MediSensePreprocessor(dataset_name)
    X_train_proc = preprocessor.fit_transform(X_train, y_train)
    X_test_proc = preprocessor.transform(X_test)
    
    return X_train_proc, y_train, X_test_proc, y_test

## Evaluate Base Learners on Each Dataset

In [ ]:
skf = StratifiedKFold(n_splits=STACKING_PARAMS['n_folds'], shuffle=True, random_state=SEED)
all_results = {}

for ds_name in ['heart', 'diabetes', 'liver']:
    print(f"\n{'=' * 60}")
    print(f"Dataset: {ds_name.upper()}")
    print(f"{'=' * 60}")
    
    X_train, y_train, X_test, y_test = prepare_dataset(ds_name)
    base_learners = get_base_learners()
    ds_results = []
    
    for name, model in base_learners:
        # Cross-validation scores
        cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')
        cv_f1 = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1')
        cv_auc = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc')
        
        # Train on full training set and evaluate on test set
        from sklearn.base import clone
        fitted = clone(model)
        fitted.fit(X_train, y_train)
        y_pred = fitted.predict(X_test)
        y_prob = fitted.predict_proba(X_test)[:, 1] if hasattr(fitted, 'predict_proba') else None
        test_metrics = compute_metrics(y_test, y_pred, y_prob)
        
        ds_results.append({
            'model': name,
            'cv_accuracy_mean': cv_scores.mean(),
            'cv_accuracy_std': cv_scores.std(),
            'cv_f1_mean': cv_f1.mean(),
            'cv_auc_mean': cv_auc.mean(),
            'test_accuracy': test_metrics['accuracy'],
            'test_f1': test_metrics['f1'],
            'test_roc_auc': test_metrics.get('roc_auc', None),
        })
        
        print(f"\n  {name.upper()}:")
        print(f"    CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
        print(f"    CV F1:       {cv_f1.mean():.4f}")
        print(f"    CV ROC-AUC:  {cv_auc.mean():.4f}")
        print(f"    Test metrics:\n{format_metrics(test_metrics)}")
    
    all_results[ds_name] = pd.DataFrame(ds_results)

## Results Summary Tables

In [ ]:
for ds_name, res_df in all_results.items():
    print(f"\n{ds_name.upper()} — Model Comparison:")
    display(res_df.set_index('model').round(4))

## Visual Comparison: CV Accuracy and Test F1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (ds_name, res_df) in enumerate(all_results.items()):
    ax = axes[idx]
    x = np.arange(len(res_df))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, res_df['cv_accuracy_mean'], width, label='CV Accuracy',
                   yerr=res_df['cv_accuracy_std'], capsize=3, alpha=0.8)
    bars2 = ax.bar(x + width/2, res_df['test_f1'], width, label='Test F1', alpha=0.8)
    
    ax.set_xlabel('Model')
    ax.set_ylabel('Score')
    ax.set_title(f'{ds_name.upper()}')
    ax.set_xticks(x)
    ax.set_xticklabels(res_df['model'], rotation=45)
    ax.legend()
    ax.set_ylim(0, 1.1)

plt.suptitle('Base Learner Comparison Across Datasets', fontsize=14)
plt.tight_layout()
plt.show()

## ROC-AUC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(5)  # 5 models
width = 0.25

for i, (ds_name, res_df) in enumerate(all_results.items()):
    ax.bar(x + i * width, res_df['cv_auc_mean'], width, label=ds_name.upper(), alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('CV ROC-AUC')
ax.set_title('Cross-Validated ROC-AUC by Model and Dataset')
ax.set_xticks(x + width)
model_names = all_results['heart']['model'].tolist()
ax.set_xticklabels(model_names, rotation=45)
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## Summary

- All five base learners have been evaluated on heart, diabetes, and liver datasets.
- Tree-based models (RF, XGBoost, LightGBM) generally perform well across all datasets.
- SVM and KNN performance varies more depending on the dataset characteristics.
- These baselines serve as the foundation for the stacking ensemble in notebook 07.